# CI failure triage: rules vs LLM agent vs hybrid
Runs on a free Colab CPU runtime. Put your Gemini key in Colab Secrets as `GEMINI_API_KEY` (key icon on the left, turn on notebook access).

In [ ]:
!git clone -q https://github.com/lassya2507/ci-failure-triage-agent.git
%cd ci-failure-triage-agent
!pip -q install -r requirements.txt
!python -m bench.fetch
!python -m pytest -q tests

In [ ]:
import os
from google.colab import userdata
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

from google import genai
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
names = [m.name.split("/")[-1] for m in client.models.list()]
print("\n".join(n for n in names if "flash" in n))

Pick the model from the list above. The default is the cheapest Flash-Lite model.

In [ ]:
MODEL = "gemini-3.1-flash-lite"
!python -m evals.run --system rules --split test

Smoke test: 3 dev cases. Check that you see `ok`/`XX` lines and no FALLBACK on every line.

In [ ]:
!python -m evals.run --system agent --split dev --limit 3 --model $MODEL

Full run on the test split. If it stops with a daily-quota message, run this cell again later; it resumes.

In [ ]:
!python -m evals.run --system agent --split test --model $MODEL

In [ ]:
!python -m evals.report --split test

In [ ]:
!zip -qr results.zip results
from google.colab import files
files.download("results.zip")